# 02 — Dynamic Factor Model

Extract 4 latent factors from the transformed FRED panel using the
DFM (PCA + EM Kalman filter).  The factors capture:

1. **Real Activity** — industrial production, income, sales
2. **Labor Market** — payrolls, unemployment, claims
3. **Inflation** — CPI, PCE, PPI
4. **Financial Conditions** — yield curve, credit spreads, equities

We validate by comparing factor 1 to CFNAI and shading NBER recessions.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.fred_client import FREDClient
from src.data.data_pipeline import DataPipeline

client = FREDClient()
pipeline = DataPipeline(fred_client=client, start_date='1979-01-01')
panel = pipeline.run(save_vintage=False)
print(f'Panel: {panel.shape[0]} months × {panel.shape[1]} series')


In [ ]:
from src.models.dynamic_factor_model import DynamicFactorModel

dfm = DynamicFactorModel(
    n_factors=4,
    factor_names=['real_activity', 'labor_market', 'inflation', 'financial_conditions'],
    max_iter=100,
)
dfm.fit(panel.dropna(how='all'))
factors = dfm.factors_
print('Factor shape:', factors.shape)
factors.tail()

In [ ]:
# NBER recessions for shading
nber_recessions = [
    ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
for ax, col, color in zip(axes, factors.columns, colors):
    factors[col].plot(ax=ax, color=color, linewidth=1.5)
    ax.set_ylabel(col.replace('_', ' ').title())
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
    for start, end in nber_recessions:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.15, color='red')
plt.suptitle('Latent Factors from DFM (shaded = NBER recessions)', fontsize=13)
plt.tight_layout()
plt.show()


## Factor 1 vs CFNAI

The first factor (real activity) should correlate strongly with the
Chicago Fed National Activity Index — both are broad economic
activity measures.


In [ ]:
if 'CFNAI' in panel.columns:
    cfnai = panel['CFNAI'].dropna()
    f1 = factors.iloc[:, 0].reindex(cfnai.index).dropna()
    cfnai_aligned = cfnai.loc[f1.index]

    corr = f1.corr(cfnai_aligned)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(f1.index, f1 / f1.std(), label=f'Factor 1 (z-score)', alpha=0.8)
    ax.plot(cfnai_aligned.index, cfnai_aligned / cfnai_aligned.std(),
            label='CFNAI (z-score)', alpha=0.8, linestyle='--')
    for start, end in nber_recessions:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.15, color='red')
    ax.set_title(f'Factor 1 vs CFNAI (correlation = {corr:.3f})')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('CFNAI not available in panel')
